# JSONL To Books Cache

Convert persisted `.jsonl` / `.jsonl.gz` venue files into `books_df` `.pkl.gz` cache files.

This notebook is useful when raw JSON replay is too slow and you want a reusable pickle input for `data_pipeline.ipynb`.


## Config

- `INPUT_ROOT`: run folder or file to ingest
- `SYMBOL`: canonical symbol filter, or `ALL`
- `OUTPUT_PATH`: optional explicit output `.pkl.gz` path
- `ANALYSIS_CACHE_ROOT`: default base dir when `OUTPUT_PATH` is not set

- `ENABLE_CHECKPOINTS`: save partial books_df after each input file
- `CHECKPOINT_PATH`: optional explicit checkpoint path


In [1]:
from pathlib import Path

INPUT_ROOT = Path("/home/fkonrad97/projects/pop/persist/pop-data/2026-03-05-20-27-39")
SYMBOL = "BTCUSDT"  # use "ALL" to keep every symbol
OUTPUT_PATH = None  # optional explicit run-summary manifest path override
ANALYSIS_CACHE_ROOT = Path("/home/fkonrad97/projects/pop/persist/analysis-cache")
ENABLE_CHECKPOINTS = True
CHECKPOINT_PATH = None  # optional explicit checkpoint .pkl.gz path
REMOVE_CHECKPOINT_AFTER_SUCCESS = True
TARGET_SHARD_BYTES = 128 * 1024 * 1024  # flush when buffered rows estimate reaches about 128MB
MAX_ROWS_PER_SHARD = 100_000  # safety cap for very light rows
MIN_ROWS_PER_SHARD = 10_000  # avoid tiny shards while the buffer is still warming up


In [2]:
from __future__ import annotations

import gzip
import json
import zlib
from decimal import Decimal, InvalidOperation
from pathlib import Path

import pandas as pd


def is_supported_input(path: Path) -> bool:
    name = path.name.lower()
    return name.endswith(".jsonl") or name.endswith(".jsonl.gz")


def expand_inputs(root: Path) -> list[Path]:
    if root.is_file():
        return [root] if is_supported_input(root) else []
    if not root.is_dir():
        return []
    return [
        candidate
        for candidate in sorted(root.rglob("*"))
        if candidate.is_file() and is_supported_input(candidate)
    ]


def open_text(path: Path):
    if path.name.lower().endswith(".gz"):
        return gzip.open(path, "rt", encoding="utf-8")
    return path.open("rt", encoding="utf-8")


def normalize_symbol(symbol: str) -> str:
    base = symbol.split("@", 1)[0]
    return "".join(ch for ch in base.upper() if ch.isalnum())


def normalize_venue_name(value: str) -> str:
    return normalize_symbol(value).lower()


def parse_decimal(value: object) -> Decimal | None:
    if value is None:
        return None
    try:
        return Decimal(str(value))
    except (InvalidOperation, ValueError):
        return None


def parse_levels(levels: object) -> list[dict[str, str]]:
    parsed: list[dict[str, str]] = []
    if not isinstance(levels, list):
        return parsed
    for raw in levels:
        if not isinstance(raw, dict):
            continue
        price_raw = str(raw.get("price", "")).strip()
        quantity_raw = str(raw.get("quantity", "")).strip()
        price = parse_decimal(price_raw)
        quantity = parse_decimal(quantity_raw)
        if price is None or quantity is None or quantity <= 0:
            continue
        parsed.append({"price": price_raw, "quantity": quantity_raw})
    return parsed


def sanitize_for_pickle(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return df.copy()
    out = df.copy()
    out.columns = [str(col) for col in out.columns]
    for column in out.columns:
        series = out[column]
        if pd.api.types.is_string_dtype(series.dtype):
            out[column] = series.astype(object)
        elif pd.api.types.is_integer_dtype(series.dtype):
            out[column] = series.astype("int64")
        elif pd.api.types.is_float_dtype(series.dtype):
            out[column] = series.astype("float64")
    return out


def extract_venue_from_path(path: Path) -> str:
    name = path.name
    if name.endswith('.jsonl.gz'):
        name = name[:-9]
    elif name.endswith('.jsonl'):
        name = name[:-6]
    return normalize_venue_name(name) or path.stem.lower()


def estimate_buffer_bytes(rows: list[dict[str, object]]) -> int:
    if not rows:
        return 0
    sample = rows[: min(len(rows), 250)]
    sample_df = pd.DataFrame(sample)
    sample_bytes = int(sample_df.memory_usage(deep=True).sum())
    scale = len(rows) / max(len(sample), 1)
    return int(sample_bytes * scale)


def should_flush_buffer(rows: list[dict[str, object]]) -> bool:
    row_count = len(rows)
    if row_count <= 0:
        return False
    if row_count >= MAX_ROWS_PER_SHARD:
        return True
    if row_count < MIN_ROWS_PER_SHARD:
        return False
    return estimate_buffer_bytes(rows) >= TARGET_SHARD_BYTES


def resolve_output_path(input_root: Path, analysis_cache_root: Path, symbol_filter: str | None) -> Path:
    if input_root.is_file():
        run_folder = input_root.parent.name
    else:
        run_folder = input_root.name
    symbol_suffix = (symbol_filter or "all_symbols").lower()
    return analysis_cache_root / run_folder / f"books_df_{symbol_suffix}_{run_folder}.manifest.pkl.gz"


def resolve_checkpoint_path(output_path: Path, explicit_checkpoint_path: Path | None) -> Path:
    if explicit_checkpoint_path is not None:
        return explicit_checkpoint_path
    run_folder = output_path.parent.name
    return output_path.parent / f"jsonl_to_books_cache_milestone_{run_folder}.pkl"


def build_shard_path(output_path: Path, venue: str, symbol_filter: str | None, part_id: int) -> Path:
    run_folder = output_path.parent.name
    symbol_suffix = (symbol_filter or "all_symbols").lower()
    return output_path.parent / f"books_df_{venue}_{symbol_suffix}_{run_folder}_part{part_id:04d}.pkl.gz"


def build_venue_manifest_path(output_path: Path, venue: str, symbol_filter: str | None) -> Path:
    run_folder = output_path.parent.name
    symbol_suffix = (symbol_filter or "all_symbols").lower()
    return output_path.parent / f"books_df_{venue}_{symbol_suffix}_{run_folder}.manifest.pkl.gz"


def load_existing_checkpoint(checkpoint_path: Path) -> dict[str, object]:
    if not checkpoint_path.exists():
        return {
            'processed_files': [],
            'current_source_path': None,
            'next_part_id': 1,
            'written_shards': [],
        }
    compression = 'gzip' if checkpoint_path.name.lower().endswith('.gz') else None
    payload = pd.read_pickle(checkpoint_path, compression=compression)
    return {
        'processed_files': list(payload.get('processed_files', [])),
        'current_source_path': payload.get('current_source_path'),
        'next_part_id': int(payload.get('next_part_id', 1)),
        'written_shards': list(payload.get('written_shards', [])),
    }


def save_checkpoint(checkpoint_path: Path, state: dict[str, object]) -> None:
    checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
    pd.to_pickle(
        {
            'processed_files': sorted(set(state.get('processed_files', []))),
            'current_source_path': state.get('current_source_path'),
            'next_part_id': int(state.get('next_part_id', 1)),
            'written_shards': list(state.get('written_shards', [])),
        },
        checkpoint_path,
        compression='gzip' if checkpoint_path.name.lower().endswith('.gz') else None,
    )


def cleanup_checkpoint(checkpoint_path: Path, remove_after_success: bool) -> bool:
    if not remove_after_success or not checkpoint_path.exists():
        return False
    checkpoint_path.unlink()
    return True


def cleanup_partial_shards_for_active_source(state: dict[str, object]) -> list[str]:
    """Remove shards for an interrupted source file and rewind the next part id.

    If the process crashes while one source file is mid-conversion, any shards already
    written for that source are deleted so the source can be replayed cleanly. After
    removal, `next_part_id` is recomputed from the remaining kept shards so resumed
    output numbering stays contiguous instead of leaving gaps.
    """
    current_source_path = state.get('current_source_path')
    if not current_source_path:
        return []
    if current_source_path in set(state.get('processed_files', [])):
        return []
    removed: list[str] = []
    kept: list[dict[str, object]] = []
    for shard in state.get('written_shards', []):
        source_path = shard.get('source_path')
        shard_path = Path(shard['path'])
        if source_path == current_source_path:
            if shard_path.exists():
                shard_path.unlink()
            removed.append(str(shard_path))
        else:
            kept.append(shard)
    state['written_shards'] = kept
    if kept:
        max_part_id = max(int(item.get('part_id', 0)) for item in kept)
        state['next_part_id'] = max_part_id + 1
    else:
        state['next_part_id'] = 1
    return removed


def build_row(obj: dict[str, object], source_path: Path) -> dict[str, object] | None:
    venue = str(obj.get('venue', '')).strip()
    symbol = str(obj.get('symbol', '')).strip()
    ts_book_ns = int(obj.get('ts_book_ns', 0))
    top_n = int(obj.get('top_n', 0))
    if not venue or not symbol or ts_book_ns <= 0:
        return None

    bids = parse_levels(obj.get('bids'))
    asks = parse_levels(obj.get('asks'))
    if not bids or not asks:
        return None

    return {
        'venue': normalize_venue_name(venue),
        'symbol': symbol,
        'canonical_symbol': normalize_symbol(symbol),
        'ts_book_ns': ts_book_ns,
        'ts_book_dt': pd.to_datetime(ts_book_ns, unit='ns'),
        'top_n': top_n,
        'source_path': str(source_path),
        'best_bid_price': bids[0]['price'],
        'best_bid_qty': bids[0]['quantity'],
        'best_ask_price': asks[0]['price'],
        'best_ask_qty': asks[0]['quantity'],
        'bid_levels': bids,
        'ask_levels': asks,
    }


def flush_rows_to_shard(rows: list[dict[str, object]], output_path: Path, venue: str, symbol_filter: str | None, part_id: int) -> tuple[Path, int]:
    shard_path = build_shard_path(output_path, venue, symbol_filter, part_id)
    shard_path.parent.mkdir(parents=True, exist_ok=True)
    shard_df = pd.DataFrame(rows)
    if not shard_df.empty:
        for column in ['best_bid_price', 'best_bid_qty', 'best_ask_price', 'best_ask_qty']:
            shard_df[column] = pd.to_numeric(shard_df[column], errors='coerce')
        shard_df.sort_values(['ts_book_ns', 'venue'], inplace=True, ignore_index=True)
    sanitize_for_pickle(shard_df).to_pickle(shard_path, compression='gzip')
    return shard_path, int(len(shard_df))


In [3]:
input_root = INPUT_ROOT.expanduser().resolve()
analysis_cache_root = ANALYSIS_CACHE_ROOT.expanduser().resolve()
symbol_filter = None if str(SYMBOL).upper() == "ALL" else normalize_symbol(SYMBOL)
output_path = OUTPUT_PATH.expanduser().resolve() if OUTPUT_PATH else resolve_output_path(input_root, analysis_cache_root, symbol_filter)
checkpoint_path = CHECKPOINT_PATH.expanduser().resolve() if CHECKPOINT_PATH else resolve_checkpoint_path(output_path, None)

input_paths = expand_inputs(input_root)
input_paths[:10]


[PosixPath('/home/fkonrad97/projects/pop/persist/pop-data/2026-03-05-20-27-39/binance.jsonl.gz'),
 PosixPath('/home/fkonrad97/projects/pop/persist/pop-data/2026-03-05-20-27-39/bybit.jsonl.gz'),
 PosixPath('/home/fkonrad97/projects/pop/persist/pop-data/2026-03-05-20-27-39/kucoin.jsonl.gz'),
 PosixPath('/home/fkonrad97/projects/pop/persist/pop-data/2026-03-05-20-27-39/okx.jsonl.gz')]

In [4]:
stats = {
    "files_seen": 0,
    "book_state_rows": 0,
    "accepted_rows": 0,
    "skipped_non_book_state": 0,
    "skipped_invalid_json": 0,
    "skipped_invalid_timestamp": 0,
    "skipped_missing_top_of_book": 0,
    "skipped_unreadable_files": 0,
    "shards_written": 0,
    "rows_written_to_shards": 0,
}

checkpoint_state = load_existing_checkpoint(checkpoint_path) if ENABLE_CHECKPOINTS else {
    'processed_files': [],
    'current_source_path': None,
    'next_part_id': 1,
    'written_shards': [],
}
removed_partial_shards = cleanup_partial_shards_for_active_source(checkpoint_state) if ENABLE_CHECKPOINTS else []
if removed_partial_shards and ENABLE_CHECKPOINTS:
    checkpoint_state['current_source_path'] = None
    save_checkpoint(checkpoint_path, checkpoint_state)

processed_files = set(checkpoint_state.get('processed_files', []))
remaining_input_paths = [path for path in input_paths if str(path) not in processed_files]

for path in remaining_input_paths:
    venue = extract_venue_from_path(path)
    checkpoint_state['current_source_path'] = str(path)
    if ENABLE_CHECKPOINTS:
        save_checkpoint(checkpoint_path, checkpoint_state)

    buffer_rows: list[dict[str, object]] = []
    try:
        stats['files_seen'] += 1
        with open_text(path) as handle:
            while True:
                try:
                    line = handle.readline()
                except (EOFError, gzip.BadGzipFile, zlib.error, OSError):
                    break
                if not line:
                    break
                line = line.strip()
                if not line:
                    continue
                try:
                    obj = json.loads(line)
                except json.JSONDecodeError:
                    stats['skipped_invalid_json'] += 1
                    continue
                if obj.get('event_type') != 'book_state':
                    stats['skipped_non_book_state'] += 1
                    continue
                stats['book_state_rows'] += 1
                row = build_row(obj, path)
                if row is None:
                    venue_raw = str(obj.get('venue', '')).strip()
                    symbol_raw = str(obj.get('symbol', '')).strip()
                    ts_book_ns = int(obj.get('ts_book_ns', 0)) if obj.get('ts_book_ns') else 0
                    bids = parse_levels(obj.get('bids'))
                    asks = parse_levels(obj.get('asks'))
                    if not venue_raw or not symbol_raw or ts_book_ns <= 0:
                        stats['skipped_invalid_timestamp'] += 1
                    elif not bids or not asks:
                        stats['skipped_missing_top_of_book'] += 1
                    continue
                if symbol_filter and row['canonical_symbol'] != symbol_filter:
                    continue
                stats['accepted_rows'] += 1
                buffer_rows.append(row)
                if should_flush_buffer(buffer_rows):
                    shard_path, shard_rows = flush_rows_to_shard(buffer_rows, output_path, venue, symbol_filter, int(checkpoint_state['next_part_id']))
                    checkpoint_state['written_shards'].append({
                        'path': str(shard_path),
                        'source_path': str(path),
                        'venue': venue,
                        'rows': shard_rows,
                        'part_id': int(checkpoint_state['next_part_id']),
                    })
                    checkpoint_state['next_part_id'] = int(checkpoint_state['next_part_id']) + 1
                    stats['shards_written'] += 1
                    stats['rows_written_to_shards'] += shard_rows
                    buffer_rows = []
                    if ENABLE_CHECKPOINTS:
                        save_checkpoint(checkpoint_path, checkpoint_state)
    except (gzip.BadGzipFile, zlib.error, OSError):
        stats['skipped_unreadable_files'] += 1
        continue

    if buffer_rows:
        shard_path, shard_rows = flush_rows_to_shard(buffer_rows, output_path, venue, symbol_filter, int(checkpoint_state['next_part_id']))
        checkpoint_state['written_shards'].append({
            'path': str(shard_path),
            'source_path': str(path),
            'venue': venue,
            'rows': shard_rows,
            'part_id': int(checkpoint_state['next_part_id']),
        })
        checkpoint_state['next_part_id'] = int(checkpoint_state['next_part_id']) + 1
        stats['shards_written'] += 1
        stats['rows_written_to_shards'] += shard_rows
        buffer_rows = []
        if ENABLE_CHECKPOINTS:
            save_checkpoint(checkpoint_path, checkpoint_state)

    processed_files.add(str(path))
    checkpoint_state['processed_files'] = sorted(processed_files)
    checkpoint_state['current_source_path'] = None
    if ENABLE_CHECKPOINTS:
        save_checkpoint(checkpoint_path, checkpoint_state)

manifest_df = pd.DataFrame(checkpoint_state.get('written_shards', []))
output_path.parent.mkdir(parents=True, exist_ok=True)
sanitize_for_pickle(manifest_df).to_pickle(output_path, compression='gzip')

venue_manifest_paths: dict[str, str] = {}
venue_shard_counts: dict[str, int] = {}
if not manifest_df.empty:
    for venue, venue_manifest_df in manifest_df.groupby('venue', sort=True):
        venue_manifest_path = build_venue_manifest_path(output_path, str(venue), symbol_filter)
        sanitize_for_pickle(venue_manifest_df.reset_index(drop=True)).to_pickle(venue_manifest_path, compression='gzip')
        venue_manifest_paths[str(venue)] = str(venue_manifest_path)
        venue_shard_counts[str(venue)] = int(len(venue_manifest_df))

checkpoint_removed = cleanup_checkpoint(checkpoint_path, REMOVE_CHECKPOINT_AFTER_SUCCESS)

{
    "input_root": str(input_root),
    "run_summary_manifest_path": str(output_path),
    "checkpoint_path": str(checkpoint_path),
    "checkpoint_enabled": ENABLE_CHECKPOINTS,
    "checkpoint_removed_after_success": checkpoint_removed,
    "processed_files": len(processed_files),
    "remaining_files": len([path for path in input_paths if str(path) not in processed_files]),
    "venue_manifest_paths": venue_manifest_paths,
    "venue_shard_counts": venue_shard_counts,
    "shard_count": int(len(manifest_df)),
    "removed_partial_shards_on_resume": len(removed_partial_shards),
    "symbol_filter": symbol_filter or "ALL",
    "stats": stats,
}


{'input_root': '/home/fkonrad97/projects/pop/persist/pop-data/2026-03-05-20-27-39',
 'run_summary_manifest_path': '/home/fkonrad97/projects/pop/persist/analysis-cache/2026-03-05-20-27-39/books_df_btcusdt_2026-03-05-20-27-39.manifest.pkl.gz',
 'checkpoint_path': '/home/fkonrad97/projects/pop/persist/analysis-cache/2026-03-05-20-27-39/jsonl_to_books_cache_milestone_2026-03-05-20-27-39.pkl',
 'checkpoint_enabled': True,
 'checkpoint_removed_after_success': True,
 'processed_files': 4,
 'remaining_files': 0,
 'venue_manifest_paths': {'binance': '/home/fkonrad97/projects/pop/persist/analysis-cache/2026-03-05-20-27-39/books_df_binance_btcusdt_2026-03-05-20-27-39.manifest.pkl.gz',
  'bybit': '/home/fkonrad97/projects/pop/persist/analysis-cache/2026-03-05-20-27-39/books_df_bybit_btcusdt_2026-03-05-20-27-39.manifest.pkl.gz',
  'kucoin': '/home/fkonrad97/projects/pop/persist/analysis-cache/2026-03-05-20-27-39/books_df_kucoin_btcusdt_2026-03-05-20-27-39.manifest.pkl.gz',
  'okx': '/home/fkonrad97

In [5]:
display("Run-level shard manifest")
display(manifest_df.head(20) if not manifest_df.empty else "No shard metadata produced.")

if venue_manifest_paths:
    display("Per-venue manifests")
    display(pd.DataFrame([
        {"venue": venue, "manifest_path": path, "shard_count": venue_shard_counts.get(venue, 0)}
        for venue, path in sorted(venue_manifest_paths.items())
    ]))


'Run-level shard manifest'

,path,source_path,venue,rows,part_id
0,/home/fkonrad97/projects/pop/persist/analysis-...,/home/fkonrad97/projects/pop/persist/pop-data/...,binance,100000,1
1,/home/fkonrad97/projects/pop/persist/analysis-...,/home/fkonrad97/projects/pop/persist/pop-data/...,binance,27085,2
2,/home/fkonrad97/projects/pop/persist/analysis-...,/home/fkonrad97/projects/pop/persist/pop-data/...,bybit,100000,3
3,/home/fkonrad97/projects/pop/persist/analysis-...,/home/fkonrad97/projects/pop/persist/pop-data/...,bybit,100000,4
4,/home/fkonrad97/projects/pop/persist/analysis-...,/home/fkonrad97/projects/pop/persist/pop-data/...,bybit,100000,5
5,/home/fkonrad97/projects/pop/persist/analysis-...,/home/fkonrad97/projects/pop/persist/pop-data/...,bybit,100000,6
6,/home/fkonrad97/projects/pop/persist/analysis-...,/home/fkonrad97/projects/pop/persist/pop-data/...,bybit,65188,7
7,/home/fkonrad97/projects/pop/persist/analysis-...,/home/fkonrad97/projects/pop/persist/pop-data/...,kucoin,100000,8
8,/home/fkonrad97/projects/pop/persist/analysis-...,/home/fkonrad97/projects/pop/persist/pop-data/...,kucoin,100000,9
9,/home/fkonrad97/projects/pop/persist/analysis-...,/home/fkonrad97/projects/pop/persist/pop-data/...,kucoin,100000,10


'Per-venue manifests'

,venue,manifest_path,shard_count
0,binance,/home/fkonrad97/projects/pop/persist/analysis-...,2
1,bybit,/home/fkonrad97/projects/pop/persist/analysis-...,5
2,kucoin,/home/fkonrad97/projects/pop/persist/analysis-...,49
3,okx,/home/fkonrad97/projects/pop/persist/analysis-...,2
